# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:**
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # metadata is an object, not a dictionary

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This step helps to understand the dataset's structure before further analysis.

We will print out the record sets (tables), their fields (columns), and their respective `@id` values.

In [ ]:
# List all available RecordSets and their field ids
record_sets = dataset.metadata.recordSet

if not record_sets:
    # Some croissant schemas provide record sets in a different way (e.g. inside 'hasPart'). Try to find them:
    if hasattr(metadata, "hasPart"):
        record_sets = [x for x in metadata.hasPart if hasattr(x, "@type") and x['@type'] == 'http://mlcommons.org/croissant/RecordSet']

if not record_sets:
    raise ValueError("No record sets found in metadata.")

for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  Name: {getattr(rs, 'name', '[no name]')}")
    print(f"  @id: {getattr(rs, '@id', '[no id]')}")
    if hasattr(rs, 'field'):
        print(f"  Fields:")
        for fld in rs.field:
            print(f"    • Name: {getattr(fld, 'name', '[no name]')}, @id: {getattr(fld, '@id', '[no id]')}")
    print()

# Save the first record set id for later usage
if hasattr(record_sets[0], '@id'):
    first_record_set_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else getattr(record_sets[0], '@id', None)
else:
    first_record_set_id = None
# List all record set ids for usage in extraction
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', '[no id]') for rs in record_sets]
print("Record Set @id(s):", record_set_ids)

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. We will be using the record set and field `@id`s found above.

You may adjust `record_set_ids` to focus on a different set/table.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded '{record_set_id}' with shape {df.shape}")
    print(f"Columns: {list(df.columns) if not df.empty else 'No columns'}\n")

# Display first few rows of the first record set
if dataframes[first_record_set_id].shape[0] > 0:
    print(dataframes[first_record_set_id].head())
else:
    print("No data loaded for the first record set.")

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, handling missing data, and grouping by key attributes. 

First, let's identify a numeric field (column) and a group field from the first record set for demonstration.

In [ ]:
from numpy import number as np_number
df = dataframes[first_record_set_id]

# Display column sample values and dtypes to help identify a numeric and group field
print("Sample columns and inferred types:\n")
print(df.dtypes)
print(df.head(2))

# Identify a numeric field (fallback to the first detected numeric type)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
# If none found, try to coerce possible fields
if not numeric_fields:
    for col in df.columns:
        try:
            coerced = pd.to_numeric(df[col])
            if coerced.notnull().any():
                numeric_fields.append(col)
        except Exception:
            continue
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Numeric field selected: {numeric_field}")
else:
    print("No numeric field detected. Unable to run numeric analysis.")
    numeric_field = None

# For group field, pick a likely categorical with limited unique values
possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() < min(8, len(df)//2)]
group_field = possible_group_fields[0] if possible_group_fields else None
print(f"Group field selected: {group_field if group_field else '[none found]'}")

# Proceed if numeric_field is available
if numeric_field:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows\n")
    if not filtered_df.empty:
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized '{numeric_field}' for filtered records (showing first few rows):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
            print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
            print(grouped_df.head())
else:
    print("Skipping EDA steps due to missing numeric field.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if available, its mean by group.

_If the numeric field could not be detected (e.g., all data are categorical), you may adjust the code to pick another column._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field}")

    if group_field:
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=axes[1])
        axes[1].set_title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the clinicopathological and molecular dataset of second primary colorectal cancer in cancer survivors using the FAIR^2 Croissant metadata standard. We:

- Loaded metadata and inspected record set structure using their `@id` values
- Extracted the primary data table to a pandas DataFrame
- Identified and demonstrated filtering, normalization, and grouping
- Visualized value distributions for key fields

This workflow can be extended by inspecting other record sets, modifying filtering/grouping as needed, and performing more detailed machine learning or statistical analyses on the data.